# Phase 2 — Data Preprocessing

## 1. Data Loading

In [45]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

## 2. Handle Missing Values

In [46]:
absence_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]

for col in absence_cols:
    train[col] = train[col].fillna("NoFeature")
    test[col] = test[col].fillna("NoFeature")

In [47]:
absence_num_cols = [
    "GarageYrBlt", "GarageCars", "GarageArea",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"
]

for col in absence_num_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

In [48]:
for df in [train, test]:
    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median())
    )

In [49]:
train["Electrical"] = train["Electrical"].fillna(train["Electrical"].mode()[0])
test["Electrical"] = test["Electrical"].fillna(train["Electrical"].mode()[0])

train["MasVnrArea"] = train["MasVnrArea"].fillna(0)
test["MasVnrArea"] = test["MasVnrArea"].fillna(0)

for df in [train, test]:
    df.loc[df["MasVnrType"].isna() & (df["MasVnrArea"] == 0), "MasVnrType"] = "NoFeature"
    df.loc[df["MasVnrType"].isna() & (df["MasVnrArea"] > 0), "MasVnrType"] = "Unknown"

In [50]:
test.loc[test["GarageYrBlt"] == 2207, "GarageYrBlt"] = 2007

## 3. Final Data Validation

In [51]:
missing = train.isna().sum().sort_values(ascending=False)
missing[missing > 0]

Series([], dtype: int64)

In [52]:
missing = test.isna().sum().sort_values(ascending=False)
missing[missing > 0]

MSZoning       4
Utilities      2
Functional     2
Exterior1st    1
Exterior2nd    1
KitchenQual    1
SaleType       1
dtype: int64

In [53]:
categorical_cols = [
    "MSZoning", "Utilities", "Functional",
    "Exterior1st", "Exterior2nd", "KitchenQual", "SaleType"
]

for col in categorical_cols:
    test[col] = test[col].fillna(train[col].mode()[0])

In [54]:
test.isna().sum().sort_values(ascending=False).head()

Id             0
MSSubClass     0
MSZoning       0
LotFrontage    0
LotArea        0
dtype: int64

In [55]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

train.to_csv("../data/processed/train_processed.csv", index=False)
test.to_csv("../data/processed/test_processed.csv", index=False)

In [56]:
print("Train:", train.shape)
print("Test :", test.shape)
print("Train IDs:", train["Id"].min(), "-", train["Id"].max())
print("Test IDs :", test["Id"].min(), "-", test["Id"].max())

Train: (1460, 81)
Test : (1459, 80)
Train IDs: 1 - 1460
Test IDs : 1461 - 2919


## 4. Preprocessing Summary

- Loaded the original train and test datasets without modifying the raw files.
- Replaced structural missing values with `"None"` for categorical features where missing means the feature is absent.
- Replaced structural numerical missing values with `0` for basement and garage features.
- Imputed `LotFrontage` using the median within `Neighborhood`.
- Imputed missing `Electrical` using the training-set mode.
- Handled `MasVnrType` according to `MasVnrArea`: `"None"` when area is 0 and `"Unknown"` when area is greater than 0.
- Replaced missing `MasVnrArea` with `0`.
- Corrected the test value `GarageYrBlt = 2207` to `2007` based on the data-quality investigation.
- Imputed the remaining test-only categorical missing values using their training-set modes.
- Verified that both train and test contain no missing values.
- Saved the processed datasets to `data/processed/`.